In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

# Notebook is located inside data/bicycle/
DATA_DIR = Path(".")

COUNTS_FILE = DATA_DIR / "Bicycle_Counts_(Historical)_20261119.csv"
COUNTERS_FILE = DATA_DIR / "Bicycle_Counters_(Historical)_20261119.csv"

OUTPUT_FILE = DATA_DIR / "bicycle_counts.csv"

print("Counts file exists:", COUNTS_FILE.exists())
print("Counters file exists:", COUNTERS_FILE.exists())

Counts file exists: True
Counters file exists: True


In [23]:
counts = pd.read_csv(COUNTS_FILE)
counters = pd.read_csv(COUNTERS_FILE)

print("Bicycle counts shape:", counts.shape)
print("Bicycle counters shape:", counters.shape)

print("\nCounts columns:")
print(counts.columns.tolist())

print("\nCounters columns:")
print(counters.columns.tolist())

/var/folders/gg/3346ph196w9g4qyr2_6lgc800000gn/T/ipykernel_35938/1772953381.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  counts = pd.read_csv(COUNTS_FILE)


Bicycle counts shape: (7378266, 5)
Bicycle counters shape: (41, 9)

Counts columns:
['countid', 'id', 'date', 'counts', 'status']

Counters columns:
['id', 'name', 'domain', 'latitude', 'longitude', 'interval', 'timezone', 'sens', 'counter']


In [24]:
display(counts.head())
display(counters.head())

print("\nStatus distribution:")
print(counts["status"].value_counts(dropna=False))

print("\nDate range:")
print(pd.to_datetime(counts["date"], errors="coerce").agg(["min", "max"]))

print("\nUnique counter IDs:", counts["id"].nunique())

,countid,id,date,counts,status
0,19831896,100009425,06/03/2023 12:00:00 AM,27,0
1,19831897,100009425,06/03/2023 12:15:00 AM,9,0
2,19831898,100009425,06/03/2023 12:30:00 AM,14,0
3,19831899,100009425,06/03/2023 12:45:00 AM,10,0
4,19831900,100009425,06/03/2023 01:00:00 AM,5,0


,id,name,domain,latitude,longitude,interval,timezone,sens,counter
0,100010017,Staten Island Ferry,New York City DOT,40.643387,-74.072075,15,(UTC-05:00) US/Eastern;DST,5,Y2H22104568
1,300027723,Fountain Ave,New York City DOT,40.655606,-73.862951,15,(UTC-05:00) US/Eastern;DST,5,Y2H22104569
2,100057318,Broadway at 50th St,New York City DOT,40.761380,-73.984080,15,(UTC-05:00) US/Eastern;DST,5,Y2H18055362
3,100010018,Pulaski Bridge,New York City DOT,40.742563,-73.951492,15,(UTC-05:00) US/Eastern;DST,5,Y2H22042948
4,100009426,Manhattan Bridge Ped Path,New York City DOT,40.714573,-73.994950,15,(UTC-05:00) US/Eastern;DST,5,Y2H22042941



Status distribution:
status
0    6077126
4    1301140
Name: count, dtype: int64

Date range:


/var/folders/gg/3346ph196w9g4qyr2_6lgc800000gn/T/ipykernel_35938/3576408563.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  print(pd.to_datetime(counts["date"], errors="coerce").agg(["min", "max"]))


min   2012-08-31 00:00:00
max   2026-01-28 04:30:00
Name: date, dtype: datetime64[ns]

Unique counter IDs: 41


In [25]:
sample = counts.copy()

sample["date"] = pd.to_datetime(
    sample["date"],
    errors="coerce"
)

counter_id = sample["id"].iloc[0]

sample_day = sample.loc[
    sample["id"] == counter_id,
    "date"
].dt.date.iloc[0]

one_day = sample[
    (sample["id"] == counter_id)
    & (sample["date"].dt.date == sample_day)
].sort_values("date")

print("Counter:", counter_id)
print("Date:", sample_day)
print("Number of observations:", len(one_day))

print("\nFirst 10 timestamps:")
print(one_day["date"].head(10))

print("\nTime between observations:")
print(one_day["date"].diff().value_counts().head())

/var/folders/gg/3346ph196w9g4qyr2_6lgc800000gn/T/ipykernel_35938/2816292701.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sample["date"] = pd.to_datetime(


Counter: 100009425
Date: 2023-06-03
Number of observations: 96

First 10 timestamps:
0   2023-06-03 00:00:00
1   2023-06-03 00:15:00
2   2023-06-03 00:30:00
3   2023-06-03 00:45:00
4   2023-06-03 01:00:00
5   2023-06-03 01:15:00
6   2023-06-03 01:30:00
7   2023-06-03 01:45:00
8   2023-06-03 02:00:00
9   2023-06-03 02:15:00
Name: date, dtype: datetime64[ns]

Time between observations:
date
0 days 00:15:00    95
Name: count, dtype: int64


In [26]:
counter_locations = counters[
    ["id", "name", "latitude", "longitude"]
].copy()

counter_locations = counter_locations.rename(
    columns={"id": "counter_id"}
)

counter_locations["counter_id"] = (
    counter_locations["counter_id"].astype(str).str.strip()
)

counter_locations["latitude"] = pd.to_numeric(
    counter_locations["latitude"],
    errors="coerce"
)

counter_locations["longitude"] = pd.to_numeric(
    counter_locations["longitude"],
    errors="coerce"
)

counter_locations = counter_locations.dropna(
    subset=["counter_id", "latitude", "longitude"]
)

counter_locations = counter_locations.drop_duplicates(
    subset=["counter_id"]
)

print("Valid counter locations:", len(counter_locations))
print("Unique counter IDs:", counter_locations["counter_id"].nunique())

display(counter_locations.head())

Valid counter locations: 41
Unique counter IDs: 41


,counter_id,name,latitude,longitude
0,100010017,Staten Island Ferry,40.643387,-74.072075
1,300027723,Fountain Ave,40.655606,-73.862951
2,100057318,Broadway at 50th St,40.761380,-73.984080
3,100010018,Pulaski Bridge,40.742563,-73.951492
4,100009426,Manhattan Bridge Ped Path,40.714573,-73.994950


In [27]:
bike_counts = counts[
    ["countid", "id", "date", "counts", "status"]
].copy()

bike_counts = bike_counts.rename(
    columns={
        "id": "counter_id",
        "counts": "count",
    }
)

bike_counts["counter_id"] = (
    bike_counts["counter_id"].astype(str).str.strip()
)

bike_counts["date"] = pd.to_datetime(
    bike_counts["date"],
    errors="coerce"
)

bike_counts["count"] = pd.to_numeric(
    bike_counts["count"],
    errors="coerce"
)

# Remove records without the necessary information
bike_counts = bike_counts.dropna(
    subset=["counter_id", "date", "count", "status"]
)

# Keep nonnegative bicycle counts
bike_counts = bike_counts[
    bike_counts["count"] >= 0
].copy()

# Keep the crash analysis period
bike_counts = bike_counts[
    bike_counts["date"].dt.year.between(2019, 2025)
].copy()

# Remove exact duplicate observation IDs, if present
print("Duplicate count IDs:", bike_counts["countid"].duplicated().sum())

bike_counts = bike_counts.drop_duplicates(
    subset=["countid"]
)

bike_counts["year"] = bike_counts["date"].dt.year

print("Clean observations:", len(bike_counts))
print("Years:", sorted(bike_counts["year"].unique()))
print("Counters:", bike_counts["counter_id"].nunique())

print("\nStatus distribution:")
print(bike_counts["status"].value_counts())

/var/folders/gg/3346ph196w9g4qyr2_6lgc800000gn/T/ipykernel_35938/606623081.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bike_counts["date"] = pd.to_datetime(


Duplicate count IDs: 0
Clean observations: 4927626
Years: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
Counters: 37

Status distribution:
status
0    4763902
4     163724
Name: count, dtype: int64


In [42]:
# Aggregate actual bicycle counts into daily totals

bike_counts["date"] = bike_counts["date"].dt.normalize()

daily_counts = (
    bike_counts.groupby(
        ["counter_id", "date", "year", "status"],
        as_index=False
    )
    .agg(
        count=("count", "sum"),  # SUM bicycle counts
        n_observations=("count", "size"),  # Number of 15-min intervals
    )
)

print("Daily records:", len(daily_counts))

display(daily_counts.head(50))

Daily records: 51651


,counter_id,date,year,status,count,n_observations
0,100009424,2019-01-01,2019,4,0.0,96
1,100009424,2019-01-02,2019,4,0.0,96
2,100009424,2019-01-03,2019,4,0.0,96
3,100009424,2019-01-04,2019,4,0.0,96
4,100009424,2019-01-05,2019,4,0.0,96
5,100009424,2019-01-06,2019,4,0.0,96
6,100009424,2019-01-07,2019,4,0.0,96
7,100009424,2019-01-08,2019,4,0.0,96
8,100009424,2019-01-09,2019,4,0.0,96
9,100009424,2019-01-10,2019,4,0.0,96


In [43]:
# Inspect one counter on one day
sample = daily_counts.iloc[0]

print("Counter:", sample["counter_id"])
print("Date:", sample["date"])
print("Status:", sample["status"])
print("Total bicycles:", sample["count"])
print("Number of observations:", sample["n_observations"])

Counter: 100009424
Date: 2019-01-01 00:00:00
Status: 4
Total bicycles: 0.0
Number of observations: 96


In [44]:
bicycle = daily_counts.merge(
    counter_locations[
        ["counter_id", "name", "latitude", "longitude"]
    ],
    on="counter_id",
    how="inner",
    validate="many_to_one",
)

print("Final daily records:", len(bicycle))
print("Unique counters:", bicycle["counter_id"].nunique())

print("\nUnmatched daily records:")
print(len(daily_counts) - len(bicycle))

display(bicycle.head())

Final daily records: 51361
Unique counters: 36

Unmatched daily records:
290


,counter_id,date,year,status,count,n_observations,name,latitude,longitude
0,100009424,2019-01-01,2019,4,0.0,96,2nd Avenue - 26th St S,40.73971,-73.97954
1,100009424,2019-01-02,2019,4,0.0,96,2nd Avenue - 26th St S,40.73971,-73.97954
2,100009424,2019-01-03,2019,4,0.0,96,2nd Avenue - 26th St S,40.73971,-73.97954
3,100009424,2019-01-04,2019,4,0.0,96,2nd Avenue - 26th St S,40.73971,-73.97954
4,100009424,2019-01-05,2019,4,0.0,96,2nd Avenue - 26th St S,40.73971,-73.97954


In [45]:
print("\nObservations per day by status:")

display(
    bicycle.groupby("status")["n_observations"]
    .describe()
)

# Percentage of counter-days with all 96 observations
completeness = (
    bicycle.assign(
        complete_day=bicycle["n_observations"] == 96
    )
    .groupby("status")["complete_day"]
    .mean()
    .mul(100)
)

print("\nPercentage of complete days:")
print(completeness)


Observations per day by status:


,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
0,49625.0,95.440343,6.717467,1.0,96.0,96.0,96.0,96.0
4,1736.0,94.311060,6.434688,15.0,96.0,96.0,96.0,96.0



Percentage of complete days:
status
0    98.875567
4    83.525346
Name: complete_day, dtype: float64


In [46]:
annual_coverage = (
    bicycle.groupby(["year", "status"])
    .agg(
        counter_days=("date", "size"),
        counters=("counter_id", "nunique"),
        average_daily_count=("count", "mean"),
        median_daily_count=("count", "median"),
    )
    .reset_index()
)

display(annual_coverage)

,year,status,counter_days,counters,average_daily_count,median_daily_count
0,2019,0,4652,15,2030.699484,1515.0
1,2019,4,940,5,2435.363830,2087.0
2,2020,0,4787,14,2655.324838,2026.0
3,2020,4,366,1,1504.355191,1420.0
4,2021,0,4942,15,2686.854310,2216.5
5,2021,4,264,2,1562.606061,1620.5
6,2022,0,6021,20,2405.392294,1934.0
7,2022,4,71,8,1949.929577,1778.0
8,2023,0,7356,28,2317.170745,1969.5
9,2023,4,95,1,4416.957895,4433.0


In [47]:
bicycle = bicycle[
    [
        "counter_id",
        "date",
        "year",
        "status",
        "count",
        "n_observations",
        "latitude",
        "longitude",
        "name",
    ]
]

bicycle.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE.resolve())
print("Final shape:", bicycle.shape)

display(bicycle.head())

Saved: /Users/jacobboyar/Desktop/Group-7-Capstone/data/bicycle/bicycle_counts.csv
Final shape: (51361, 9)


,counter_id,date,year,status,count,n_observations,latitude,longitude,name
0,100009424,2019-01-01,2019,4,0.0,96,40.73971,-73.97954,2nd Avenue - 26th St S
1,100009424,2019-01-02,2019,4,0.0,96,40.73971,-73.97954,2nd Avenue - 26th St S
2,100009424,2019-01-03,2019,4,0.0,96,40.73971,-73.97954,2nd Avenue - 26th St S
3,100009424,2019-01-04,2019,4,0.0,96,40.73971,-73.97954,2nd Avenue - 26th St S
4,100009424,2019-01-05,2019,4,0.0,96,40.73971,-73.97954,2nd Avenue - 26th St S


In [48]:
print("========== BICYCLE DATA SUMMARY ==========")

print("\nYears:")
print(sorted(bicycle["year"].unique()))

print("\nNumber of counters:")
print(bicycle["counter_id"].nunique())

print("\nNumber of daily records:")
print(len(bicycle))

print("\nStatus codes:")
print(bicycle["status"].value_counts())

print("\nAverage daily bicycle counts by year and status:")
display(
    bicycle.groupby(["year", "status"])["count"]
    .mean()
    .round(2)
    .unstack()
)

print("\nSaved file:")
print(OUTPUT_FILE.resolve())

========== BICYCLE DATA SUMMARY ==========

Years:
[2019, 2020, 2021, 2022, 2023, 2024, 2025]

Number of counters:
36

Number of daily records:
51361

Status codes:
status
0    49625
4     1736
Name: count, dtype: int64

Average daily bicycle counts by year and status:


status,0,4
year,,
2019,2030.70,2435.36
2020,2655.32,1504.36
2021,2686.85,1562.61
2022,2405.39,1949.93
2023,2317.17,4416.96
2024,1924.07,NaN
2025,1958.60,NaN



Saved file:
/Users/jacobboyar/Desktop/Group-7-Capstone/data/bicycle/bicycle_counts.csv
